In [ ]:
%pip install -q -e .. matplotlib

# 02 — Subset and plot

Pick a region of interest, fetch only that subset from the EDR server,
and plot it.  This is the workflow most users actually want.

## 1. Discover collections

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

collections = httpx.get(f"{server}/collections").raise_for_status().json()["collections"]
for c in collections:
    print(c["id"], "-", c.get("title", ""))

In [ ]:
collection_id = collections[0]["id"]  # or pick any id from the list above
collection_url = f"{server}/collections/{collection_id}"

## 2. Open with a bounding box

We narrow the request to Spain.  Only this region is fetched from the
server when we later read `.values`.

In [ ]:
import xarray as xr

import edr_xarray  # registers engine="edr"

spain = (-9.5, 36.0, 3.3, 43.8)  # (lon_min, lat_min, lon_max, lat_max)

ds = xr.open_dataset(collection_url, engine="edr", bbox=spain)
ds

## 3. Plot a timestep

Pick a single timestamp with `.sel(t=...)` to get a 2-D field, then
use xarray's built-in `.plot()` (matplotlib under the hood).

In [ ]:
var = next(iter(ds.data_vars))

# Pick a timestamp from the collection's time axis.
# Replace with any date string in the collection's range.
timestamp = str(ds.t.values[0])[:10]  # e.g. "2024-01-01"

ds[var].sel(t=timestamp).plot(figsize=(10, 6), cmap="viridis")

## 4. Cleanup

In [ ]:
ds.close()